# 02 — `Protocol`, `TypedDict`, `Literal`, `Final`

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- approfondir `Protocol` : attributs, méthodes, `@runtime_checkable`
- typer un dictionnaire de forme fixe avec `TypedDict`
- restreindre une valeur à un ensemble fini avec `Literal`
- marquer une constante avec `Final`
- distinguer `TypedDict` total vs `total=False`

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- type hints modernes (notebook 01)
- classes, `ABC`, `Protocol` de base (jour 1)

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- `Generic[T]` et `TypeVar` (notebook 03)
- `@dataclass` et son typage (section 03)

## Plan

1. Protocol approfondi
2. `TypedDict` total
3. `TypedDict` avec `total=False` et `NotRequired`
4. `Literal` : valeurs restreintes
5. `Final` : constantes typées
6. `Self` : le type de la classe
7. Synthèse
8. Exercices

---

## 1. `Protocol` approfondi

Un `Protocol` peut contenir des **attributs typés** en plus de méthodes. C'est le moyen idiomatique de décrire une structure duck-typée avec son interface complète.

In [ ]:
from typing import Protocol


class Identifiable(Protocol):
    id: int
    nom: str

    def slug(self) -> str: ...


In [ ]:
class Salle:
    def __init__(self, id: int, nom: str) -> None:
        self.id = id
        self.nom = nom
    def slug(self) -> str:
        return self.nom.lower().replace(' ', '-')


In [ ]:
def afficher(obj: Identifiable) -> None:
    print(f'#{obj.id} : {obj.nom} ({obj.slug()})')

afficher(Salle(1, 'Salle Mars'))


---

## 2. `TypedDict` total

Un `TypedDict` est un dictionnaire dont les **clés** et leurs **types** sont connus à l'avance. C'est idéal pour typer des JSON, des configurations, des messages.

In [ ]:
from typing import TypedDict


class ReservationPayload(TypedDict):
    salle: str
    creneau: str
    organisateur: str


In [ ]:
r: ReservationPayload = {
    'salle': 'Mars',
    'creneau': 'lundi 9h',
    'organisateur': 'Alice',
}
r['salle']


À l'**exécution**, `ReservationPayload` est un `dict` ordinaire. Le typage est purement statique.

---

## 3. `TypedDict` avec `total=False` et `NotRequired`

Par défaut, toutes les clés sont **obligatoires**. Trois options pour les rendre facultatives.

In [ ]:
from typing import TypedDict, NotRequired

class Config(TypedDict):
    host: str
    port: int
    debug: NotRequired[bool]  # optionnelle


In [ ]:
c1: Config = {'host': 'localhost', 'port': 5432}
c2: Config = {'host': 'prod', 'port': 5432, 'debug': True}
c1, c2


### Ou via `total=False` (tout optionnel)

In [ ]:
class ConfigPartielle(TypedDict, total=False):
    host: str
    port: int

patch: ConfigPartielle = {'port': 6543}


---

## 4. `Literal` : valeurs restreintes

`Literal` permet d'énumérer les valeurs possibles. Parfait pour des enums légers ou des paramètres à choix fermé.

In [ ]:
from typing import Literal


def aligner(texte: str, sens: Literal['gauche', 'droite', 'centre']) -> str:
    return f'[{sens}] {texte}'


In [ ]:
aligner('Bonjour', 'gauche')


`mypy` refusera `aligner('Bonjour', 'haut')` en statique. À l'exécution, rien n'empêche, mais votre contrat est explicite.

---

## 5. `Final` : constantes typées

`Final` signale qu'une valeur ne doit **jamais être réaffectée**. `mypy` attrape toute tentative de modification.

In [ ]:
from typing import Final

VERSION: Final = '1.0.0'
MAX_TENTATIVES: Final[int] = 3


In [ ]:
VERSION, MAX_TENTATIVES


À l'exécution, Python n'empêche pas la réaffectation. `Final` est un contrat pour `mypy` et pour le lecteur.

---

## 6. `Self` : le type de la classe (PEP 673)

Dans une méthode, `Self` désigne la classe courante, même dans les sous-classes. Évite d'écrire `'MaClasse'` en chaîne (ou d'importer `TypeVar` à la main).

In [ ]:
from typing import Self


class Chainable:
    def __init__(self, parts: list[str] | None = None) -> None:
        self.parts = parts or []
    def add(self, p: str) -> Self:
        self.parts.append(p)
        return self


In [ ]:
Chainable().add('a').add('b').add('c').parts


L'intérêt de `Self` : si `SousChainable(Chainable)` existe, `add` renvoie automatiquement `SousChainable`, pas `Chainable`.

---

## Synthèse

| Outil | Rôle |
|---|---|
| `Protocol` | Interface structurelle (attributs + méthodes) |
| `TypedDict` | Dict avec clés et types fixes |
| `NotRequired`, `total=False` | Rendre certaines clés facultatives |
| `Literal[...]` | Restreindre à un ensemble de valeurs |
| `Final` | Constante non réassignable |
| `Self` | Type de la classe courante |


### Règles à retenir

1. **`Protocol` avant `ABC`** quand on n'a pas besoin d'imposer un héritage.
2. **`TypedDict` pour du JSON typé** ; `@dataclass` pour un vrai objet Python.
3. **`Literal` pour un set fermé de chaînes** ; `Enum` pour un set fermé de valeurs **sémantiques** (voir section 06).
4. **`Final` clarifie l'intention** même si Python ne l'empêche pas à l'exécution.
5. **`Self` > chaîne de caractères** dans les signatures qui renvoient `self`.

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — `TypedDict` basique *(facile)*

Définir un `TypedDict` `Personne` avec `nom: str`, `age: int`, `email: str`. Écrire une fonction `formater(p: Personne) -> str` qui renvoie `'Nom <email> (age ans)'`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Protocol_typeddict", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
from typing import TypedDict

class Personne(TypedDict):
    nom: str
    age: int
    email: str

def formater(p: Personne) -> str:
    return f"{p['nom']} <{p['email']}> ({p['age']} ans)"

print(formater({'nom': 'Alice', 'age': 30, 'email': 'alice@ex.fr'}))
```

</details>

### Exercice 2 — `Literal` + fonction *(moyen)*

Écrire `convertir(valeur: float, sens: Literal['eur_vers_usd', 'usd_vers_eur']) -> float` utilisant un taux fixe de 1.10. Vérifier les deux sens.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Protocol_typeddict", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
from typing import Literal

TAUX = 1.10

def convertir(valeur: float, sens: Literal['eur_vers_usd', 'usd_vers_eur']) -> float:
    if sens == 'eur_vers_usd':
        return valeur * TAUX
    return valeur / TAUX

print(convertir(100, 'eur_vers_usd'))
print(convertir(100, 'usd_vers_eur'))
```

</details>

### Exercice 3 — `Protocol` avec attributs *(moyen)*

Définir un `Protocol` `HasName` avec un attribut `nom: str`. Écrire `trier_par_nom(objets: list[HasName]) -> list[HasName]`. Tester avec deux classes différentes.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Protocol_typeddict", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
from typing import Protocol

class HasName(Protocol):
    nom: str

def trier_par_nom(objets: list[HasName]) -> list[HasName]:
    return sorted(objets, key=lambda o: o.nom)

class Personne:
    def __init__(self, nom: str) -> None:
        self.nom = nom
    def __repr__(self) -> str:
        return f'P({self.nom})'

class Ville:
    def __init__(self, nom: str) -> None:
        self.nom = nom
    def __repr__(self) -> str:
        return f'V({self.nom})'

print(trier_par_nom([Personne('Charlie'), Ville('Alpha'), Personne('Bob')]))
```

</details>

### Exercice 4 — `TypedDict` avec clés optionnelles *(difficile)*

Définir `EventPayload` avec :

- `name: str` (obligatoire)
- `timestamp: str` (obligatoire)
- `user_id: int` (optionnel via `NotRequired`)
- `context: dict[str, str]` (optionnel).

Écrire `log_event(e: EventPayload) -> str` qui renvoie une ligne textuelle.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Protocol_typeddict", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
from typing import TypedDict, NotRequired

class EventPayload(TypedDict):
    name: str
    timestamp: str
    user_id: NotRequired[int]
    context: NotRequired[dict[str, str]]

def log_event(e: EventPayload) -> str:
    parts = [e['timestamp'], e['name']]
    if 'user_id' in e:
        parts.append(f'uid={e["user_id"]}')
    if 'context' in e:
        parts.append(str(e['context']))
    return ' | '.join(parts)

print(log_event({'name': 'LOGIN', 'timestamp': '2026-04-14T09:00'}))
print(log_event({'name': 'ERROR', 'timestamp': '2026-04-14T10:00', 'user_id': 42, 'context': {'ip': '1.2.3.4'}}))
```

</details>

---

## Ressources externes

### Documentation officielle
- [`typing.TypedDict`](https://docs.python.org/3/library/typing.html#typing.TypedDict)
- [`typing.Literal`](https://docs.python.org/3/library/typing.html#typing.Literal)
- [`typing.Protocol`](https://docs.python.org/3/library/typing.html#typing.Protocol)

### PEPs de référence
- **PEP 544** — Protocols
- **PEP 586** — Literal types
- **PEP 589** — TypedDict
- **PEP 591** — Final
- **PEP 655** — Required / NotRequired
- **PEP 673** — `Self` type